<a href="https://colab.research.google.com/github/KayraOrbay/GROUP37_CENG313/blob/eda-guncelleme/assignment1_eda_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CENG313 PROJECT

***Problem Definition***


This project is based on the competion in the Kaggle *Image Matching Challenge 2025*. Our main goal is to estimate 3D camera poses by identifying reliable matching points between 2D images of the same scene, which are captured from various viewpoints and lighting conditions.

Instead of a traditional tabular target, the dataset targets rotation matrices and translation vectors between images. These matrices will be calculated using matched keypoints.


The model's performance is measured using the mAA (Mean Average Accuracy) metric. This metric evaluates the accuracy of our predicted camera poses by calculating the rotation and translation errors compared to the ground truth.

In [ ]:
print("Colab çalışıyor")

Colab çalışıyor


In [ ]:
%pip install -q uv

In [ ]:
!git clone https://github.com/KayraOrbay/GROUP37_CENG313.git

fatal: destination path 'GROUP37_CENG313' already exists and is not an empty directory.


In [ ]:
%cd /content/GROUP37_CENG313
!ls

/content/GROUP37_CENG313
assignment1_eda_baseline.ipynb	GROUP37_CENG313  requirements.txt
data				README.md	 src


In [ ]:
import subprocess
import sys
from pathlib import Path

requirements = Path("requirements.txt")
assert requirements.is_file(), "Repo klasöründe requirements.txt bulunamadı."

subprocess.run(
    [
        sys.executable, "-m", "uv", "pip", "install",
        "--system",
        "--python", sys.executable,
        "-r", str(requirements),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'uv', 'pip', 'install', '--system', '--python', '/usr/bin/python3', '-r', 'requirements.txt'], returncode=0)

In [ ]:
# Proje yapisi ve kurulum kontrolu
from pathlib import Path
import sys

repo_root = Path('/content/GROUP37_CENG313')
assert (repo_root / 'src').is_dir(), 'src klasoru bulunamadi.'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import src
print('src import basarili')
print('src dosyalari:', [str(p.relative_to(repo_root)) for p in (repo_root / 'src').rglob('*') if p.is_file() and '__pycache__' not in str(p)])
for name in ['README.md', 'requirements.txt', '.gitignore']:
    p = repo_root / name
    print('\n---', name, '---')
    print(p.read_text()[:12000] if p.exists() else 'Dosya yok')

src import basarili
src dosyalari: ['src/data.py', 'src/_init_.py', 'src/evaluation.py', 'src/models.py']

--- README.md ---
# image-matching-challenge-2025
"https://www.kaggle.com/competitions/image-matching-challenge-2025"

Ayşegül Pınar Yılmaz 
23118080004
Cherrylin

Kayra Orbay
23118080017 
KayraOrbay


--- requirements.txt ---


--- .gitignore ---
# Datasets and downloaded archives
data/
datasets/
train_labels.csv
train_thresholds.csv
sample_submission.csv
*.zip

# Model weights
weights/
checkpoints/
*.pt
*.pth
*.ckpt
*.h5
*.onnx
*.safetensors

# Credentials
.env
.env.*
!.env.example
secrets/
kaggle.json
credentials.json
service-account*.json
*.pem
*.key

# Python and notebook temporary files
.venv/
__pycache__/
*.pyc
.ipynb_checkpoints/



In [ ]:
# Gercek proje modullerini kontrol et
import importlib
for module_name in ['src.data', 'src.models', 'src.evaluation']:
    module = importlib.import_module(module_name)
    print(module_name, 'OK:', module.__file__)
    print(Path(module.__file__).read_text()[:8000] or '(Dosya bos)')

# Veri, model agirliklari ve erisim bilgilerini Git takibinden haric tut
ignore_path = repo_root / '.gitignore'
existing = ignore_path.read_text() if ignore_path.exists() else ''
rules = ['data/', 'datasets/', 'weights/', 'checkpoints/', '*.pt', '*.pth', '*.ckpt', '*.h5', '*.onnx', '*.safetensors', '*.zip', '.env', '.env.*', '!.env.example', 'secrets/', 'kaggle.json', 'credentials.json', 'service-account*.json', '*.pem', '*.key', '__pycache__/', '*.pyc', '.ipynb_checkpoints/']
missing = [rule for rule in rules if rule not in existing.splitlines()]
if missing:
    ignore_path.write_text(existing + ('\n' if existing and not existing.endswith('\n') else '') + '\n'.join(missing) + '\n')
print('\n.gitignore hazir:', ignore_path)
print('Veri klasorleri:', [str(p) for p in [repo_root / 'data', Path('/content/data'), Path('/content/drive/MyDrive')] if p.exists()])

src.data OK: /content/GROUP37_CENG313/src/data.py
from pathlib import Path
import numpy as np
import pandas as pd

REQUIRED_COLUMNS = ["dataset", "scene", "image", "rotation_matrix", "translation_vector"]


def load_train_labels(data_dir="data"):
    """Read train_labels.csv as-is (no rows dropped)."""
    df = pd.read_csv(Path(data_dir) / "train_labels.csv")
    missing = set(REQUIRED_COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"train_labels.csv is missing columns: {missing}")
    return df


def load_train_thresholds(data_dir="data"):
    return pd.read_csv(Path(data_dir) / "train_thresholds.csv")


def load_sample_submission(data_dir="data"):
    return pd.read_csv(Path(data_dir) / "sample_submission.csv")


def _parse_vector(value, expected_len):
    """'a;b;c' -> np.array([a, b, c]). NaN -> array of NaN."""
    if pd.isna(value):
        return np.full(expected_len, np.nan)
    parts = [p for p in str(value).split(";") if p != ""]
    arr = np.array([float

In [ ]:
# Kaggle train_labels.csv dosyasini oku; ek paket gerektirmez.
from pathlib import Path
import importlib
import sys
from google.colab import files

repo_root = Path('/content/GROUP37_CENG313')
assert (repo_root / 'src').is_dir(), 'Once repo klonlama ve kurulum hucrelerini calistirin.'
loader_source = '''import csv
from pathlib import Path

def read_train_labels(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"dataset", "scene", "image", "rotation_matrix", "translation_vector"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError("Beklenen train_labels.csv sutunlari eksik.")
        rows = list(reader)
    if not rows:
        raise ValueError("Etiket dosyasi bos.")
    return rows
'''
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import src.data
importlib.reload(src.data)

data_dir = repo_root / 'data'
data_dir.mkdir(exist_ok=True)
labels_path = data_dir / 'train_labels.csv'
if not labels_path.exists():
    print('Kaggle sitesinden indirilen train_labels.csv dosyasini secin.')
    uploaded = files.upload()
    if 'train_labels.csv' not in uploaded:
        raise ValueError('train_labels.csv secilmedi.')
    labels_path.write_bytes(uploaded['train_labels.csv'])

rows = src.data.read_train_labels(labels_path)
print('Veri okuma basarili. Satir sayisi:', len(rows))
print('Sutunlar:', list(rows[0]))
print('Ilk kayit:', rows[0])
print('Veri kumesi sayisi:', len({row['dataset'] for row in rows}))

Veri okuma basarili. Satir sayisi: 1945
Sutunlar: ['dataset', 'scene', 'image', 'rotation_matrix', 'translation_vector']
Ilk kayit: {'dataset': 'imc2023_haiper', 'scene': 'fountain', 'image': 'fountain_image_116.png', 'rotation_matrix': '0.122655949;0.947713775;-0.294608417;0.122670643;0.280096125;0.952101924;0.984838784;-0.152920769;-0.081901202', 'translation_vector': '0.093771314;-0.803560988;2.062001533'}
Veri kumesi sayisi: 13


In [37]:
%%writefile /content/GROUP37_CENG313/src/data.py
from pathlib import Path
import numpy as np
import pandas as pd

REQUIRED_COLUMNS = ["dataset", "scene", "image", "rotation_matrix", "translation_vector"]

def load_train_labels(data_dir="data"):
    df = pd.read_csv(Path(data_dir) / "train_labels.csv")
    missing = set(REQUIRED_COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"train_labels.csv is missing columns: {missing}")
    return df

def load_train_thresholds(data_dir="data"):
    return pd.read_csv(Path(data_dir) / "train_thresholds.csv")

def load_sample_submission(data_dir="data"):
    return pd.read_csv(Path(data_dir) / "sample_submission.csv")

def _parse_vector(value, expected_len):
    if pd.isna(value):
        return np.full(expected_len, np.nan)
    parts = [p for p in str(value).split(";") if p != ""]
    arr = np.array([float(p) for p in parts], dtype=float)
    if arr.size != expected_len:
        raise ValueError(f"Expected {expected_len} values, got {arr.size}: {value!r}")
    return arr

def parse_poses(df):
    out = df.copy()
    R = np.vstack(out["rotation_matrix"].apply(_parse_vector, expected_len=9).to_numpy())
    t = np.vstack(out["translation_vector"].apply(_parse_vector, expected_len=3).to_numpy())
    out[[f"R{i}{j}" for i in range(3) for j in range(3)]] = R
    out[["tx", "ty", "tz"]] = t
    return out

Overwriting /content/GROUP37_CENG313/src/data.py


In [45]:
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 40)

# Data Shape
display(Markdown("### 1. Data shape"))
shape_table = pd.DataFrame(
    {"rows": [df.shape[0] for df in (labels, thresholds, sample_sub)],
     "columns": [df.shape[1] for df in (labels, thresholds, sample_sub)]},
    index=["train_labels", "train_thresholds", "sample_submission"])
display(shape_table)

# Datasets / Scenes
display(Markdown("### 2. Datasets and scenes"))
print("Unique datasets:", labels["dataset"].nunique())
print("Unique (dataset, scene) pairs:", labels.groupby(["dataset", "scene"]).ngroups)

# Data Types
display(Markdown("### 3. Data types (train_labels)"))
types_table = pd.DataFrame({
    "dtype": labels.dtypes.astype(str),
    "non_null": labels.notna().sum(),
    "unique": labels.nunique(),
    "example": labels.iloc[0].astype(str).str.slice(0, 40),
})
display(types_table)

# Missing values
display(Markdown("### 4. Missing values"))
# Missing poses can be real NaN cells or text like "nan;nan;nan"
empty_cells = labels.isna()
nan_text = labels.apply(lambda col: col.astype(str).str.fullmatch(r"(?i)nan(;nan)*")) & ~empty_cells
missing_table = pd.DataFrame({
    "empty (NaN)": empty_cells.sum(),
    "'nan;nan;...' text": nan_text.sum(),
    "total missing": (empty_cells | nan_text).sum(),
    "missing_%": ((empty_cells | nan_text).mean() * 100).round(2),
})
display(missing_table)

# Duplicates
display(Markdown("### 5. Duplicates"))
duplicate_table = pd.DataFrame({"count": [
    labels.duplicated().sum(),
    labels.duplicated(["dataset", "image"]).sum(),
    labels["image"].duplicated().sum(),
]}, index=["fully duplicated rows",
           "duplicated (dataset, image)",
           "same image name in another dataset"])
display(duplicate_table)

# describe()
display(Markdown("### 6. Summary statistics – describe()"))
summary = labels.describe(include="all").T
summary["top"] = summary["top"].astype(str).str.slice(0, 40)
display(summary)


### 1. Data shape

,rows,columns
train_labels,1945,5
train_thresholds,30,3
sample_submission,1945,6


### 2. Datasets and scenes

Unique datasets: 13
Unique (dataset, scene) pairs: 34


### 3. Data types (train_labels)

,dtype,non_null,unique,example
dataset,object,1945,13,imc2023_haiper
scene,object,1945,30,fountain
image,object,1945,1937,fountain_image_116.png
rotation_matrix,object,1945,1824,0.122655949;0.947713775;-0.294608417...
translation_vector,object,1945,1824,0.093771314;-0.803560988;2.062001533


... (2 outputs left out)

### 5. Duplicates

,count
fully duplicated rows,0
"duplicated (dataset, image)",0
same image name in another dataset,8


### 6. Summary statistics – describe()

,count,unique,top,freq
dataset,1945,13,pt_brandenburg_british_buckingham,225
scene,1945,30,peach,200
image,1945,1937,dioscuri_archive_0026.png,2
rotation_matrix,1945,1824,nan;nan;nan;nan;nan;nan;nan;nan;nan,122
translation_vector,1945,1824,nan;nan;nan,122


***Data Overview***

1. Data shape

In the results of the previous code, we see that train_labels.csv has 1945 rows and 5 columns, so we have 1945 training images in total. sample_submission.csv also has 1945 rows which shows us that the submission file covers the same images, just in a different format. train_thresholds.csv has 30 rows and when we look at it we can say that each row belongs to one scene.

2. Datasets and scenes

We see that the images come from 13 datasets and [X] (dataset, scene) pairs. Looking at these, we can say that one dataset can include more than one scene. For example, pt_brandenburg_british_buckingham has 3 different scenes. This tells us that the task is not only estimating camera poses; we also need to figure out which images belong to the same scene.

3. Data types

In this table we see that all columns are object which means they are stored as text. This is expected for dataset, scene and image but we notice that rotation_matrix and translation_vector are also text. When we look at the example values, we see that they are numbers separated by ; (9 values for rotation, 3 for translation). So we can say that before doing any numerical analysis we first need to convert these strings into numeric arrays.

4. Missing values

Here we see that dataset, scene and image have no missing values. However, rotation_matrix and translation_vector are missing in [122] rows. When we checked which rows these are we noticed that all of them belong to the outliers scene. From this we can say that these are not really "missing data". These images do not belong to any scene, so they have no camera pose on purpose. Since finding these outlier images is also part of the challenge, we will only leave them out only when we calculate pose statistics.

5. Duplicates

In this table we see that there are no fully duplicated rows and no duplicated (dataset, image) pairs, so every image appears only once inside its dataset. However, when we look only at the image names, we see that [8] image names appear in two different datasets. When we checked these rows, we noticed that they come from the dioscuri scene, which is included both in imc2023_heritage and imc2024_dioscuri_baalshamin. Because of this, we can say that the image name alone is not enough to identify an image, so we will use the (dataset, image) pair as a unique key.

6. Summary statistics

From the describe() results we see that every value in the image column is unique, which matches what we found in the duplicates part. On the other hand, dataset and scene are not evenly distributed. The most frequent dataset is [top dataset] with [freq] images and the most frequent scene is [top scene] with [freq] images. Looking at this, we can say that some scenes have many more images than others, so we need to look at this imbalance in more detail in the EDA part. We also see that the most frequent value in the rotation_matrix and translation_vector columns is the nan pattern, which again points to the outlier images we found in the missing values part.